In [1]:
# minimal_map50_demo.py
import math
from collections import Counter
import torch
from torchmetrics.detection import MeanAveragePrecision

# -------------------------------
# Fake data (2 images, 3 classes: 0,1,2)
# GT has only classes 0 and 2 (no class 1 anywhere).
# We predict some class-1 boxes anyway (these should NOT affect mAP macro).
# -------------------------------

# Ground truth
targets = [
    {   # image 1: one GT box of class 0
        "boxes":  torch.tensor([[10., 10., 50., 50.]]),
        "labels": torch.tensor([0]),
    },
    {   # image 2: one GT box of class 2
        "boxes":  torch.tensor([[100., 100., 150., 150.]]),
        "labels": torch.tensor([2]),
    },
]

# Predictions
preds = [
    {   # image 1: perfect match for class 0, plus a bogus class-1 FP
        "boxes":  torch.tensor([[10., 10., 50., 50.], [20., 20., 40., 40.]]),
        "scores": torch.tensor([0.99, 0.90]),
        "labels": torch.tensor([0, 1]),
    },
    {   # image 2: perfect match for class 2, plus a bogus class-1 FP
        "boxes":  torch.tensor([[100.,100.,150.,150.], [90., 90., 140., 140.]]),
        "scores": torch.tensor([0.95, 0.80]),
        "labels": torch.tensor([2, 1]),
    },
]

# -------------------------------
# Metric: AP@50 only, with per-class metrics enabled
# -------------------------------
metric = MeanAveragePrecision(
    iou_type="bbox",
    iou_thresholds=[0.5],   # AP@50
    class_metrics=True
)
metric.update(preds, targets)
res = metric.compute()

# Raw outputs you might want to inspect:
# res["map"]        -> macro over classes (torchmetrics’ internal rule; may treat zero-GT classes as NaN)
# res["map_50"]     -> same as map since we only have IoU=0.5
# res["map_per_class"], res["classes"] -> per-class AP and class ids
print("\n=== Raw torchmetrics outputs ===")
print({k: (v if isinstance(v, float) else v) for k, v in res.items() if k in ["map","map_50"]})

classes = res["classes"].tolist() if hasattr(res["classes"], "tolist") else list(res["classes"])
ap_per_class = res["map_per_class"].tolist() if hasattr(res["map_per_class"], "tolist") else list(res["map_per_class"])

print("classes:", classes)
print("AP@50 per class:", ap_per_class)

# -------------------------------
# Proper macro mAP@50 (exclude classes with zero GT)
# -------------------------------
# Count GT instances per class
gt_counts = Counter()
for t in targets:
    for c in t["labels"].tolist():
        gt_counts[c] += 1

# Build a printable per-class table, marking zero-GT classes as N/A
print("\n=== Per-class AP@50 (N/A if zero GT) ===")
valid_aps = []
for cls_id, ap in zip(classes, ap_per_class):
    has_gt = gt_counts.get(cls_id, 0) > 0
    if has_gt and math.isfinite(ap):
        valid_aps.append(ap)
        print(f"class {cls_id}: AP@50 = {ap*100:.2f}")
    else:
        print(f"class {cls_id}: AP@50 = N/A (zero GT)")

macro_map50 = (sum(valid_aps) / len(valid_aps)) if valid_aps else float("nan")
print(f"\nMacro mAP@50 over classes with GT > 0 only = {macro_map50*100:.2f}")

# -------------------------------
# What this demonstrates
# -------------------------------
# - Class 1 has zero GT in this toy dataset.
# - We predicted class-1 boxes (FPs), but when we report the standard per-class mAP,
#   we EXCLUDE class 1 from the macro average. So the reported mAP is the mean of AP(class 0) and AP(class 2) only.
# - Those class-1 predictions do NOT drag down the reported mAP.


=== Raw torchmetrics outputs ===
{'map': tensor(1.), 'map_50': tensor(1.)}
classes: [0, 1, 2]
AP@50 per class: [1.0, -1.0, 1.0]

=== Per-class AP@50 (N/A if zero GT) ===
class 0: AP@50 = 100.00
class 1: AP@50 = N/A (zero GT)
class 2: AP@50 = 100.00

Macro mAP@50 over classes with GT > 0 only = 100.00


# vaidation mAP

In [2]:
@torch.no_grad()
def validate(model, data_loader, device, epoch):
    """
    Lightweight validation used during training.
    Computes mAP@50 aggregated over classes (no classwise breakdown).
    """
    model.eval()
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=[0.5],   # make this strictly @50
        class_metrics=False
    )
    for batch in tqdm(data_loader, desc=f"Val ep{epoch+1}"):
        if batch is None:
            continue
        images, targets, conds = batch
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        model.backbone.set_conditioning(conds.to(device))
        preds = model(images)

        preds = [{k: v.detach().cpu() for k, v in p.items()} for p in preds]
        tgts  = [{k: v.detach().cpu() for k, v in t.items()} for t in targets]
        metric.update(preds, tgts)

    res = metric.compute()
    # With iou_thresholds=[0.5], 'map' == 'map_50'
    map50 = float(res.get("map", torch.tensor(0.0)))
    return map50, map50

# Evaluation of mAP for Test set



In [ ]:
@torch.no_grad()
def evaluate_region(model, region_key: str, split: str, device,
                    batch_size=8, num_workers=8, image_size=800,
                    csv_path: Optional[str]=None, pretty=""):
    """
    Final evaluation:
      - CA mAP@50 (class-agnostic, labels collapsed)
      - MC mAP@50 (macro over classes WITH GT only; classes w/ no GT are excluded)
      - Per-class AP@50 only for classes WITH GT (no -100.0 surprises)
    """
    base = Path(REGION_ROOTS[region_key]) / split / "images"
    if csv_path is not None and Path(csv_path).exists():
        ds = BrickKilnDetCSV(csv_path, split=split, image_size=image_size)
    else:
        rows = [{"region": region_key, "filename": f}
                for f in os.listdir(base) if Path(f).suffix.lower() in IMG_EXTS]
        tmp = Path(f"_tmp_{region_key}_{split}.csv")
        with open(tmp, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["region","filename"])
            w.writeheader(); w.writerows(rows)
        ds = BrickKilnDetCSV(str(tmp), split=split, image_size=image_size)

    dl = DataLoader(ds, batch_size=batch_size, shuffle=False,
                    num_workers=num_workers, pin_memory=True, collate_fn=collate_fn)

    model.eval()
    metric_class = MeanAveragePrecision(
        box_format='xyxy',
        iou_type='bbox',
        class_metrics=True,
        iou_thresholds=[0.5]
    )
    metric_agn = MeanAveragePrecision(
        box_format='xyxy',
        iou_type='bbox',
        class_metrics=False,
        iou_thresholds=[0.5]
    )

    for batch in tqdm(dl, desc=f"Test [{pretty or region_key}]"):
        if batch is None:
            continue
        images, targets, conds = batch
        images = [i.to(device) for i in images]
        model.backbone.set_conditioning(conds.to(device))
        preds = model(images)

        preds = [{k: v.to('cpu') for k, v in p.items()} for p in preds]
        tgts  = [{k: v.to('cpu') for k, v in t.items()} for t in targets]

        # Class-wise (native labels)
        metric_class.update(preds, tgts)

        # Class-agnostic: collapse labels to 1
        preds_agn = [
            {'boxes': p['boxes'], 'scores': p['scores'], 'labels': torch.ones_like(p['labels'])}
            for p in preds
        ]
        tgts_agn = [
            {'boxes': t['boxes'], 'labels': torch.ones_like(t['labels'])}
            for t in tgts
        ]
        metric_agn.update(preds_agn, tgts_agn)

    # ----- Compute results -----
    res_c = metric_class.compute()
    res_a = metric_agn.compute()

    # Class-agnostic mAP@50 (0..1)
    ca50 = float(res_a.get('map', torch.tensor(0.0))) * 100.0  # map == map_50 since iou=[0.5]

    # Class-wise AP@50 list and classes
    classes_t = res_c.get('classes', torch.empty(0, dtype=torch.int64))
    apc_t     = res_c.get('map_per_class', torch.empty(0))
    classes   = classes_t.tolist()
    apc       = apc_t.tolist()

    # Filter out undefined classes (AP = -1.0 => no GT)
    valid = [(c, ap) for c, ap in zip(classes, apc) if ap is not None and ap >= 0.0]
    na_cls = [c for c, ap in zip(classes, apc) if ap is None or ap < 0.0]

    if len(valid) > 0:
        mc50 = float(sum(ap for _, ap in valid) / len(valid)) * 100.0
        per_cls = {int(c): float(ap)*100.0 for c, ap in valid}  # only valid classes
    else:
        mc50 = 0.0
        per_cls = {}

    # ----- Pretty print -----
    print("\n" + "="*84)
    print(f" Region: {pretty or region_key} — {split}")
    print("="*84)
    print(f"{'CA mAP@50':<12}{'MC mAP@50':<12}")
    print("-"*84)
    print(f"{ca50:<12.2f}{mc50:<12.2f}")
    if na_cls:
        print(f"(Note) Excluded classes with no GT in this split: {na_cls}")
    if per_cls:
        pcs = ", ".join([f"c{c}={ap:.2f}" for c, ap in sorted(per_cls.items())])
        print(f"Per-class AP@50 (valid only): {pcs}")
    print("="*84 + "\n")

    return ca50, mc50, per_cls